<a href="https://colab.research.google.com/github/bvvnmanikanta/Scaler-AI-Notebooks/blob/main/class06_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 6 · n8n: Build Your First AI Automation · class notebook

*AI & Agents: From Talking to AI to Building One · Scaler · Rishav Singh · v7, 19 Nov 2026*

**First: File → Save a copy in Drive.** Then this copy is yours to edit.

**How to use it.** When a slide says **Run along**, **Build along**, **Live demo** or **Hands-on** with *Notebook · Section N*, come to that section here.
- **To copy a prompt or a field:** click inside its code cell, press **Ctrl+A** then **Ctrl+C** (Cmd on a Mac). Or run the cell and copy the printed text.
- Every code cell runs in Colab as it is, with no API key. Two cells talk to the internet and skip by themselves if they can't (Section 7's news fetch and Section 9's webhook call).

**Today in one line:** data comes in → the AI works on it → an action goes out, every time, with nobody typing.

## Section 0 · Before class: accounts and tabs

1. **n8n Cloud:** sign up at **n8n.io** → free trial. It's 14 days with no card, and includes 1,000 executions (one execution = one full run). Your workspace gets an address ending in **app.n8n.cloud**. When the trial ends, n8n deletes the workspace; you then have 90 days to download your workflows.
2. **Gemini API key:** **aistudio.google.com/apikey** → Create API key → copy it. The free tier is enough for today. (Free-tier prompts may be used by Google to improve its products, so don't paste anything private.)
3. **A personal Gmail.** College or office Google accounts often block the "Sign in with Google" step.
4. *Optional, for Challenge A:* a **Telegram bot.** In Telegram, open **@BotFather** → `/newbot` → give it a name, then a username ending in `bot` → copy the token it gives you. Keep the token secret: anyone with it controls your bot.

**Key won't work in class?** On n8n Cloud, the Gemini model's credential list has **Use Gateway credits**: AI with no key, paid from your trial's credits.

## Section 1 · Run along: open n8n and find the doorbell *(slide 6 · 4 min)*

1. Open your n8n workspace → **Overview** → **Create workflow**. You get a blank canvas with a dotted square.
2. Click **Add first step**. A panel asks **"What triggers this workflow?"**
3. Read the list. Every option is a **doorbell**: something that starts the workflow (a click, a schedule, a form, a webhook call, an event in another app).
4. **Post in chat:** the trigger our standup form needs. Don't click it yet.

**Watch for:** the three building blocks. Trigger = the doorbell. Node = one step, one job. Connection = the wire the data travels on.

## Section 2 · Build along 1: the standup form *(slide 13 · 8 min)*

**Steps**
1. **Add first step** → **On form submission** (search "n8n Form" if you don't see it).
2. **Form Title:** `Daily Standup`.
3. **Form Elements** → add the three fields below. Type the labels **exactly** (copy them from the next cell). Leave **Field Name** empty, so the label becomes the box name.
4. Click **Execute step**. n8n opens a test form in a new tab ("This is a test version of your form"). Fill it with `TEST_ANSWERS` and submit.
5. Back in n8n, look at the **OUTPUT** panel. Try the **Schema**, **Table** and **JSON** views.

**Post in chat:** you asked 3 questions. How many boxes came out?

**Watch for:** the box names are your labels, letter for letter. That matters in part 2.

*Pro move:* in the OUTPUT panel, click **Pin data**. Now part 2 can re-run without you submitting the form again (pinned data is used only while you test, never in real runs).

In [ ]:
FORM_TITLE = "Daily Standup"

# (label, element type) — type the labels exactly as written
FIELDS = [
    ("Your Name", "Text Input"),
    ("What did you ship today", "Textarea"),
    ("What are the blockers", "Textarea"),
]

TEST_ANSWERS = {
    "Your Name": "Priya",
    "What did you ship today": "Login API with tests",
    "What are the blockers": "Need settings design",
}

# A second update for after you Publish (slide 15): tests the "No blockers" line
SECOND_ANSWERS = {
    "Your Name": "Arjun",
    "What did you ship today": "Fixed the flaky checkout test",
    "What are the blockers": "None",
}

for label, kind in FIELDS:
    assert label == label.strip(), f"no spaces at the ends of {label!r}"
    print(f"{label:<26} {kind}")
print()
print("Test answers:", TEST_ANSWERS)

## Section 3 · Build along 2: the AI step *(slide 14 · 12 min)*

The cue card's AI step is "a system prompt + an input". We use the **Basic LLM Chain**: one model call, the same path every run. (The **AI Agent** box adds memory and tools and picks its own next step: that's Class 7.)

**Steps**
1. Click **+** on the right of the form step → search **Basic LLM Chain**.
2. Under the chain, click the **Model** + → **Google Gemini Chat Model** → Credential: **Create new** → paste your key → **Save**. Model: pick **`models/gemini-3.5-flash-lite`** (free; Google's pick for summarising). Any current Flash model on the list works. Don't pick 2.5-flash: it's closed to new keys.
3. Back in the chain: **Source for Prompt (User Message)** → **Define below**.
4. **Prompt (User Message):** drag the three boxes from the **INPUT** panel and type the words around them. When you're done it should read like `USER_MESSAGE` below. Drag, don't type the box names.
5. **Chat Messages (if Using a Chat Model)** → **Add prompt** → Type: **System** → paste `SYSTEM_MESSAGE`.
6. **Execute step.** The summary appears in OUTPUT as a box called `text`.

**Post in chat:** change "under 50 words" to "under 20", Execute step again, paste your line 1. Then put it back to 50.

**Watch for:** only this step runs. If the summary is too long or vague, that's a prompt problem, not an n8n problem: fix the prompt, re-run this one step.

In [ ]:
SYSTEM_MESSAGE = """You are an assistant that summarises daily standup updates.

You will receive a team member's standup submission with their name,
what they shipped, and what's blocking them.

Your task:
Write a brief 3-line summary:
Line 1: Who submitted and their top accomplishment
Line 2: Key blocker (or "No blockers" if none)
Line 3: One suggested action for the team lead

Keep it under 50 words. No greetings, no sign-offs."""

# What the Prompt (User Message) field looks like after you drag the three boxes in
USER_MESSAGE = '{{ $json["Your Name"] }} achieved {{ $json["What did you ship today"] }}. Blockers: {{ $json["What are the blockers"] }}'

print(SYSTEM_MESSAGE)
print()
print(USER_MESSAGE)

### Section 3b · Why "drag, don't type": the undefined bug, in Python

n8n fills each `{{ $json["…"] }}` with the box of that exact name. The cell below does the same with a tiny stand-in. Prompt B is the cue card's own line: its names don't match the form's labels (capital N, question mark), so those spots come back empty. n8n's preview shows them as `[undefined]`.

In [ ]:
import re

def fill(template, item):
    """A tiny stand-in for n8n's expressions: swap each {{ $json["name"] }} for item["name"]."""
    return re.sub(r'\{\{ \$json\["(.+?)"\] \}\}', lambda m: item.get(m.group(1), "[undefined]"), template)

ITEM = {"Your Name": "Priya", "What did you ship today": "Login API with tests", "What are the blockers": "Need settings design"}

PROMPT_A = '{{ $json["Your Name"] }} achieved {{ $json["What did you ship today"] }}'
PROMPT_B = '{{ $json["Your name"] }} achieved {{ $json["What did you ship today?"] }}'   # the cue card's line

print("A:", fill(PROMPT_A, ITEM))
print("B:", fill(PROMPT_B, ITEM))

assert fill(PROMPT_A, ITEM) == "Priya achieved Login API with tests"
assert "[undefined]" in fill(PROMPT_B, ITEM)

## Section 4 · Build along 3: Gmail, then Publish *(slide 15 · 10 min)*

**Steps**
1. Click **+** after the AI step → **Gmail** → **Send a message**. Credential: **Sign in with Google**, allow the permissions.
2. **To:** your own email address.
3. **Subject:** type `Standup: ` then scroll the INPUT panel to the form step and drag **Your Name** after it.
4. **Email Type:** **Text** (the default, HTML, loses the line breaks).
5. **Message:** drag **text** from the AI step.
6. *Optional:* Options → turn off **Append n8n Attribution** (otherwise every email ends with "This email was sent automatically with n8n").
7. **Execute step** → check your inbox.
8. **Publish** (top right). Edits save by themselves; Publish makes it live. **Unpublish** is in the menu next to Publish.
9. Open the form step → switch to **Production URL** → copy it → open it on your phone → submit `SECOND_ANSWERS` (or your own update).

**Post in chat:** ✅ when the email lands.

**Watch for:** the **Test URL** only listens while you click Execute; the **Production URL** works after Publish, with your laptop closed. Real runs show up under **Executions**, not on the canvas.

In [ ]:
GMAIL = {
    "To": "your own address",
    "Subject": 'Standup: {{ $(\'On form submission\').item.json["Your Name"] }}',
    "Email Type": "Text",
    "Message": "{{ $json.text }}",
}
for k, v in GMAIL.items():
    print(f"{k:<11} {v}")

### Section 4b · Catch-up: paste the finished workflow

Fell behind? A workflow is just JSON. Run the next cell, copy everything it prints, click an empty spot on a **blank** n8n canvas, press **Ctrl+V** (Cmd+V on a Mac). The three steps and the Gemini model appear, already wired. Then:
1. Open **Google Gemini Chat Model** → pick your Gemini credential (or create one, or **Use Gateway credits**).
2. Open **Send a message** → pick your Google sign-in → change **To** to your address.
3. **Execute workflow** → submit the test form → the email arrives.

No keys or logins are inside the JSON, so it's safe to share.

In [ ]:
import json

CATCH_UP_WORKFLOW = '{\n  "name": "Daily standup summary",\n  "nodes": [\n    {\n      "parameters": {\n        "formTitle": "Daily Standup",\n        "formFields": {\n          "values": [\n            {\n              "fieldLabel": "Your Name",\n              "requiredField": true\n            },\n            {\n              "fieldLabel": "What did you ship today",\n              "fieldType": "textarea",\n              "requiredField": true\n            },\n            {\n              "fieldLabel": "What are the blockers",\n              "fieldType": "textarea"\n            }\n          ]\n        },\n        "options": {}\n      },\n      "type": "n8n-nodes-base.formTrigger",\n      "typeVersion": 2.5,\n      "position": [\n        0,\n        0\n      ],\n      "id": "a1b2c3d4-0001-4c55-9a51-000000000001",\n      "name": "On form submission",\n      "webhookId": "a1b2c3d4-0001-4c55-9a51-0000000000f1"\n    },\n    {\n      "parameters": {\n        "promptType": "define",\n        "text": "={{ $json[\\"Your Name\\"] }} achieved {{ $json[\\"What did you ship today\\"] }}. Blockers: {{ $json[\\"What are the blockers\\"] }}",\n        "messages": {\n          "messageValues": [\n            {\n              "type": "SystemMessagePromptTemplate",\n              "message": "You are an assistant that summarises daily standup updates.\\n\\nYou will receive a team member\'s standup submission with their name,\\nwhat they shipped, and what\'s blocking them.\\n\\nYour task:\\nWrite a brief 3-line summary:\\nLine 1: Who submitted and their top accomplishment\\nLine 2: Key blocker (or \\"No blockers\\" if none)\\nLine 3: One suggested action for the team lead\\n\\nKeep it under 50 words. No greetings, no sign-offs."\n            }\n          ]\n        }\n      },\n      "type": "@n8n/n8n-nodes-langchain.chainLlm",\n      "typeVersion": 1.9,\n      "position": [\n        260,\n        0\n      ],\n      "id": "a1b2c3d4-0001-4c55-9a51-000000000002",\n      "name": "Basic LLM Chain"\n    },\n    {\n      "parameters": {\n        "modelName": "models/gemini-3.5-flash-lite",\n        "options": {}\n      },\n      "type": "@n8n/n8n-nodes-langchain.lmChatGoogleGemini",\n      "typeVersion": 1.2,\n      "position": [\n        240,\n        220\n      ],\n      "id": "a1b2c3d4-0001-4c55-9a51-000000000003",\n      "name": "Google Gemini Chat Model"\n    },\n    {\n      "parameters": {\n        "resource": "message",\n        "operation": "send",\n        "sendTo": "you@gmail.com",\n        "subject": "=Standup: {{ $(\'On form submission\').item.json[\\"Your Name\\"] }}",\n        "emailType": "text",\n        "message": "={{ $json.text }}",\n        "options": {\n          "appendAttribution": false\n        }\n      },\n      "type": "n8n-nodes-base.gmail",\n      "typeVersion": 2.1,\n      "position": [\n        620,\n        0\n      ],\n      "id": "a1b2c3d4-0001-4c55-9a51-000000000004",\n      "name": "Send a message"\n    }\n  ],\n  "connections": {\n    "On form submission": {\n      "main": [\n        [\n          {\n            "node": "Basic LLM Chain",\n            "type": "main",\n            "index": 0\n          }\n        ]\n      ]\n    },\n    "Google Gemini Chat Model": {\n      "ai_languageModel": [\n        [\n          {\n            "node": "Basic LLM Chain",\n            "type": "ai_languageModel",\n            "index": 0\n          }\n        ]\n      ]\n    },\n    "Basic LLM Chain": {\n      "main": [\n        [\n          {\n            "node": "Send a message",\n            "type": "main",\n            "index": 0\n          }\n        ]\n      ]\n    }\n  }\n}'

wf = json.loads(CATCH_UP_WORKFLOW)
names = {n["name"] for n in wf["nodes"]}
for src, outs in wf["connections"].items():
    assert src in names
    for kind in outs.values():
        for branch in kind:
            for link in branch:
                assert link["node"] in names
assert all("credentials" not in n for n in wf["nodes"])   # no logins or keys inside
print(CATCH_UP_WORKFLOW)

## Section 5 · Hands-on: trace it, then break it *(slides 17 and 19)*

### Exercise 5.1 · Trace the data (5 min)
1. Click the **form step**: in OUTPUT, find the exact field name for "What did you ship today".
2. Click the **AI step**: look at its INPUT and its prompt. Does the name in the prompt match check 1, letter for letter? If not, that's a bug waiting to happen.
3. Click the **Gmail step**: its INPUT should show the AI's summary text.

Tip: **Executions** (left sidebar, or the tab above the canvas) shows every real run. Open the one from your phone: every step's data is there. A failed run has **Debug in editor**, which loads that run's data into the canvas.

**Post in chat:** "match" or "mismatch", and the first words Gmail received.

### Exercise 5.2 · Break it on purpose (7 min)
One break at a time. **Before each run, post RED or GREEN in chat:** will a step turn red? Undo each break before the next (Ctrl+Z, or paste the original back from Section 3).

In [ ]:
BREAKS = [
    ("1 Wrong field name", "In the AI step's prompt, replace one dragged box with the typed text  {{ $json.favourite_color }}"),
    ("2 Empty prompt",     "Delete the System message in the AI step"),
    ("3 Wrong address",    "In Gmail, set To = not-an-email"),
]
for name, how in BREAKS:
    print(f"{name}\n   {how}\n   Before you run it: RED or GREEN?\n")

**My results** *(double-click to type)*

| Break | Red or green? | What did the output or the email say? |
|---|---|---|
| 1 Wrong field name | | |
| 2 Empty prompt | | |
| 3 Wrong address | | |

The cue card's lessons: 1 → silent wrong values (hardest to catch); 2 → an AI step with no prompt is a function with no parameters; 3 → real errors are red and tell you why.

## Section 6 · Live demo: two words vs a schema *(slide 24 · watch)*

Challenge B's AI answer goes **straight into a spreadsheet cell**, so the format must be exact. The cue card's prompt asks for "ONLY two words". Rishav runs it on three messages, then turns on a **schema**.

**What Rishav does**
1. Basic LLM Chain + Gemini, system message = `TWO_WORDS_PROMPT`, user message = one of `MESSAGES`.
2. Messages 1 and 2 give clean answers. **Predict before message 3:** what lands in the category cell?
3. Turn on **Require Specific Output Format** → connect a **Structured Output Parser** → **Schema Type: Define using JSON Schema** → paste `SCHEMA`. Swap the system message for `SCHEMA_PROMPT`.
4. Run message 3 again: the step outputs `output.category` and `output.priority`, one allowed value each. If the model ever breaks the shape, the step goes **red** instead of writing junk into your sheet.

This is Class 3's "please return JSON" and Class 4's structured output, inside a workflow step.

In [ ]:
import json

TWO_WORDS_PROMPT = """Read the text in the "message" column.
Categorise it as one of: bug, feature_request, praise, complaint, or question.
Also rate the priority as: low, medium, high, or urgent.
Reply with ONLY two words on one line: the category and the priority.
Example: bug high"""

MESSAGES = [
    "The app crashes when I upload a photo.",
    "Can you add CSV export to the reports page?",
    "Love the new dark mode! But PDF export has been broken since Monday and my report is due tomorrow.",
]

SCHEMA_PROMPT = """Read the customer message.
Pick the ONE category that fits best: bug, feature_request, praise, complaint or question.
If the message mixes several, pick the one the team must act on.
Rate the priority: low, medium, high or urgent."""

SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "bug",
                "feature_request",
                "praise",
                "complaint",
                "question"
            ]
        },
        "priority": {
            "type": "string",
            "enum": [
                "low",
                "medium",
                "high",
                "urgent"
            ]
        }
    },
    "required": [
        "category",
        "priority"
    ]
}

print(TWO_WORDS_PROMPT, "", *MESSAGES, "", SCHEMA_PROMPT, "", json.dumps(SCHEMA, indent=2), sep="\n")

### Section 6b · Offline: why two words breaks and a schema doesn't

The cell checks answers the way your sheet would use them. The "loose" answers are the kind of thing the two-words prompt sends back; message 3 is the usual failure (sample answers, written for this check).

In [ ]:
CATEGORIES = SCHEMA["properties"]["category"]["enum"]
PRIORITIES = SCHEMA["properties"]["priority"]["enum"]

def from_two_words(text):
    """What a 'split on the space' step does with the AI's reply."""
    parts = text.strip().split()
    if len(parts) != 2:
        raise ValueError(f"expected 2 words, got {len(parts)}: {text!r}")
    return {"category": parts[0], "priority": parts[1]}

def check(obj):
    """The schema's rule: both keys present, each one allowed value."""
    assert set(obj) == {"category", "priority"}, obj
    assert obj["category"] in CATEGORIES, obj
    assert obj["priority"] in PRIORITIES, obj
    return obj

LOOSE = ["bug high", "feature_request low", "praise, bug urgent"]   # sample replies to MESSAGES
WITH_SCHEMA = [{"category": "bug", "priority": "high"},
               {"category": "feature_request", "priority": "low"},
               {"category": "bug", "priority": "urgent"}]

for msg, reply in zip(MESSAGES, LOOSE):
    try:
        print("OK  ", check(from_two_words(reply)))
    except (ValueError, AssertionError) as e:
        print("FAIL", e)

print()
for obj in WITH_SCHEMA:
    print("OK  ", check(obj))

## Section 7 · Mini build A: the Morning Briefing *(slide 25)*

**Schedule Trigger → HTTP Request → Basic LLM Chain (+ Gemini) → Telegram.** It's Class 5's retrieve-then-generate, on a timer.

**Steps**
1. **Add first step → On a schedule** (Schedule Trigger): Trigger Interval **Days**, hour **8**. Workflow **…** menu → **Settings** → Timezone **Asia/Kolkata**, or "8 am" may be 8 am somewhere else.
2. **+ → HTTP Request**: Method GET, URL = `NEWS_URL`. Execute step: the output has a list called `hits` (title, url, points…).
   *Why not the cue card's `topstories.json`?* It returns only story numbers, so the AI would have nothing to read.
3. **+ → Basic LLM Chain** (+ Google Gemini Chat Model): Define below, Prompt (User Message) = `NEWS_EXPRESSION`, system message = `BRIEFING_PROMPT` (the cue card's, word for word).
4. **+ → Telegram → Send a text message**: create the credential with your bot token; **Chat ID** = yours (see `get_chat_id_steps` below); **Text** = drag `text` from the AI step. Additional Fields → **Append n8n Attribution** off.
5. Test with **Execute workflow** (don't wait for 8 am). Then **Publish**.

**Done when:** your phone buzzes with exactly 3 clean lines.

In [ ]:
NEWS_URL = "https://hn.algolia.com/api/v1/search?tags=front_page"

# Paste into the Prompt (User Message) field (switch the field to Expression if needed)
NEWS_EXPRESSION = "{{ $json.hits.slice(0, 20).map(h => '- ' + h.title + ' ' + (h.url || 'https://news.ycombinator.com/item?id=' + h.objectID)).join('\\n') }}"

BRIEFING_PROMPT = """You are a tech news summariser for a busy software engineer.

You will receive a list of today's top stories (titles and URLs).

Your task:
Pick the 3 stories most relevant to a working software engineer.
Ignore celebrity news, politics, and clickbait.

Output format — STRICT:
Exactly 3 lines. Each line formatted as:
  - 2-sentence summary followed by the URL

No greeting. No header. No sign-off. Just the three bullets."""

get_chat_id_steps = [
    "1. In Telegram, send your new bot any message (say hi).",
    "2. In a browser open: https://api.telegram.org/bot<YOUR_TOKEN>/getUpdates",
    "3. Find \"chat\":{\"id\": 123456789 ...} and copy that number. That's your Chat ID.",
    "   (If the list is empty, send the bot another message and refresh.)",
]
print(NEWS_URL, "", NEWS_EXPRESSION, "", BRIEFING_PROMPT, "", *get_chat_id_steps, sep="\n")

### Section 7b · Optional: see what the AI step will read

The cell fetches today's front page (in Colab) and builds the same text block the n8n expression builds. No internet? It uses a small sample instead.

In [ ]:
import json, urllib.request

SAMPLE = {"hits": [
    {"title": "Show HN: A tiny Postgres queue", "url": "https://example.com/pgq", "objectID": "1"},
    {"title": "Ask HN: How do you review AI-written code?", "url": None, "objectID": "2"},
]}

try:
    with urllib.request.urlopen(NEWS_URL, timeout=8) as r:
        data = json.load(r)
    print("Live front page:", len(data["hits"]), "stories\n")
except Exception as e:
    data = SAMPLE
    print("Offline, using the sample:", type(e).__name__, "\n")

def as_lines(d, n=20):
    """Python twin of NEWS_EXPRESSION."""
    return "\n".join("- " + h["title"] + " " + (h.get("url") or "https://news.ycombinator.com/item?id=" + h["objectID"]) for h in d["hits"][:n])

text = as_lines(data)
assert text.startswith("- ")
print(text)

## Section 8 · Mini build B: the Spreadsheet Sorter *(slide 25)*

**Google Sheets Trigger → Basic LLM Chain (+ Gemini + Structured Output Parser) → Google Sheets (update the row).** Quiz 2's idea, with the schema from the demo.

**Steps**
1. Make a Google Sheet called **Customer messages** with three column headers in row 1: `message`, `category`, `priority`.
2. Run the next cell, copy the JSON, click a blank canvas, **Ctrl+V**.
3. Open **Google Sheets Trigger** → Sign in with Google → pick your sheet and tab (event: **Row added**). Open **Update row in sheet** → same sheet; match on `row_number`.
4. Open **Google Gemini Chat Model** → your credential.
5. Type a message in a new row (use one of `MESSAGES` from Section 6). Click **Execute workflow**; the trigger checks for new rows on a timer (about once a minute), so wait. Then **Publish**.

**Done when:** you add a row and, within about a minute, category and priority fill in by themselves.

*Paste failed?* Build it by hand: the steps match the JSON's node names. The cue card's original two-words prompt is `TWO_WORDS_PROMPT` in Section 6.

In [ ]:
import json

SORTER_WORKFLOW = '{\n  "name": "Spreadsheet sorter (Challenge B)",\n  "nodes": [\n    {\n      "parameters": {\n        "pollTimes": {\n          "item": [\n            {\n              "mode": "everyMinute"\n            }\n          ]\n        },\n        "documentId": {\n          "__rl": true,\n          "value": "",\n          "mode": "list"\n        },\n        "sheetName": {\n          "__rl": true,\n          "value": "",\n          "mode": "list"\n        },\n        "event": "rowAdded",\n        "options": {}\n      },\n      "type": "n8n-nodes-base.googleSheetsTrigger",\n      "typeVersion": 1,\n      "position": [\n        0,\n        0\n      ],\n      "id": "b1b2c3d4-0002-4c55-9a51-000000000001",\n      "name": "Google Sheets Trigger"\n    },\n    {\n      "parameters": {\n        "promptType": "define",\n        "text": "={{ $json.message }}",\n        "hasOutputParser": true,\n        "messages": {\n          "messageValues": [\n            {\n              "type": "SystemMessagePromptTemplate",\n              "message": "Read the customer message.\\nPick the ONE category that fits best: bug, feature_request, praise, complaint or question.\\nIf the message mixes several, pick the one the team must act on.\\nRate the priority: low, medium, high or urgent."\n            }\n          ]\n        }\n      },\n      "type": "@n8n/n8n-nodes-langchain.chainLlm",\n      "typeVersion": 1.9,\n      "position": [\n        260,\n        0\n      ],\n      "id": "b1b2c3d4-0002-4c55-9a51-000000000002",\n      "name": "Basic LLM Chain"\n    },\n    {\n      "parameters": {\n        "modelName": "models/gemini-3.5-flash-lite",\n        "options": {}\n      },\n      "type": "@n8n/n8n-nodes-langchain.lmChatGoogleGemini",\n      "typeVersion": 1.2,\n      "position": [\n        200,\n        220\n      ],\n      "id": "b1b2c3d4-0002-4c55-9a51-000000000003",\n      "name": "Google Gemini Chat Model"\n    },\n    {\n      "parameters": {\n        "schemaType": "manual",\n        "inputSchema": "{\\n  \\"type\\": \\"object\\",\\n  \\"properties\\": {\\n    \\"category\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\n        \\"bug\\",\\n        \\"feature_request\\",\\n        \\"praise\\",\\n        \\"complaint\\",\\n        \\"question\\"\\n      ]\\n    },\\n    \\"priority\\": {\\n      \\"type\\": \\"string\\",\\n      \\"enum\\": [\\n        \\"low\\",\\n        \\"medium\\",\\n        \\"high\\",\\n        \\"urgent\\"\\n      ]\\n    }\\n  },\\n  \\"required\\": [\\n    \\"category\\",\\n    \\"priority\\"\\n  ]\\n}"\n      },\n      "type": "@n8n/n8n-nodes-langchain.outputParserStructured",\n      "typeVersion": 1.3,\n      "position": [\n        380,\n        220\n      ],\n      "id": "b1b2c3d4-0002-4c55-9a51-000000000004",\n      "name": "Structured Output Parser"\n    },\n    {\n      "parameters": {\n        "operation": "update",\n        "documentId": {\n          "__rl": true,\n          "value": "",\n          "mode": "list"\n        },\n        "sheetName": {\n          "__rl": true,\n          "value": "",\n          "mode": "list"\n        },\n        "columns": {\n          "mappingMode": "defineBelow",\n          "value": {\n            "row_number": "={{ $(\'Google Sheets Trigger\').item.json.row_number }}",\n            "category": "={{ $json.output.category }}",\n            "priority": "={{ $json.output.priority }}"\n          },\n          "matchingColumns": [\n            "row_number"\n          ],\n          "schema": []\n        },\n        "options": {}\n      },\n      "type": "n8n-nodes-base.googleSheets",\n      "typeVersion": 4.7,\n      "position": [\n        640,\n        0\n      ],\n      "id": "b1b2c3d4-0002-4c55-9a51-000000000005",\n      "name": "Update row in sheet"\n    }\n  ],\n  "connections": {\n    "Google Sheets Trigger": {\n      "main": [\n        [\n          {\n            "node": "Basic LLM Chain",\n            "type": "main",\n            "index": 0\n          }\n        ]\n      ]\n    },\n    "Google Gemini Chat Model": {\n      "ai_languageModel": [\n        [\n          {\n            "node": "Basic LLM Chain",\n            "type": "ai_languageModel",\n            "index": 0\n          }\n        ]\n      ]\n    },\n    "Structured Output Parser": {\n      "ai_outputParser": [\n        [\n          {\n            "node": "Basic LLM Chain",\n            "type": "ai_outputParser",\n            "index": 0\n          }\n        ]\n      ]\n    },\n    "Basic LLM Chain": {\n      "main": [\n        [\n          {\n            "node": "Update row in sheet",\n            "type": "main",\n            "index": 0\n          }\n        ]\n      ]\n    }\n  }\n}'

wf = json.loads(SORTER_WORKFLOW)
names = {n["name"] for n in wf["nodes"]}
assert all(link["node"] in names for outs in wf["connections"].values() for kind in outs.values() for branch in kind for link in branch)
parser = next(n for n in wf["nodes"] if n["name"] == "Structured Output Parser")
assert json.loads(parser["parameters"]["inputSchema"]) == SCHEMA
print("SHEET COLUMNS: message | category | priority\n")
print(SORTER_WORKFLOW)

## Section 9 · Mini build C: the Event Alert *(slide 25)*

**Any trigger → Basic LLM Chain (+ Gemini) → Gmail (or Telegram).** The fastest pick: your Gmail credential already works.

**Steps (the quick way)**
1. New workflow → **Add first step → Trigger manually**.
2. **+ → Basic LLM Chain** (+ Gemini): Define below; Prompt (User Message) = paste `SAMPLE_EVENT_TEXT`; system message = `ALERT_PROMPT` (the cue card's).
3. **+ → Gmail → Send a message** to yourself; Message = drag `text`; Email Type: Text.
4. **Execute workflow** → read the alert in your inbox.

**For engineers: let your app ring the doorbell.** Replace the manual trigger with a **Webhook** node (HTTP Method **POST**). Change the prompt to `{{ JSON.stringify($json.body) }}`. Click **Listen for test event**, copy the **Test URL** into `WEBHOOK_URL` below, run the cell: it POSTs the event as JSON, the way a signup service would. After **Publish**, use the **Production URL** instead.

**Done when:** a 2-sentence alert with a next step lands in your inbox.

In [ ]:
import json, urllib.request

SAMPLE_EVENT = {
    "event": "New signup",
    "user_name": "Priya Sharma",
    "plan": "free_trial",
    "signup_date": "2026-05-12",
    "source": "Google search",
    "first_action": "Created a project",
}
SAMPLE_EVENT_TEXT = "\n".join(f"{k}: {v}" for k, v in SAMPLE_EVENT.items())

ALERT_PROMPT = """You received some data. Write a 2-sentence summary explaining
what happened and what the most useful next step would be.
Be specific and actionable."""

print(SAMPLE_EVENT_TEXT, "", ALERT_PROMPT, sep="\n")

WEBHOOK_URL = ""   # paste your Webhook node's Test URL (or Production URL after Publish)

if WEBHOOK_URL:
    req = urllib.request.Request(WEBHOOK_URL, data=json.dumps(SAMPLE_EVENT).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=15) as r:
        print("\nn8n answered:", r.status, r.read()[:200])
else:
    print("\nNo WEBHOOK_URL yet, so nothing was sent. Paste your Test URL above and run again.")

## Section 10 · Homework and links

- **Assignment** (`Class_06_n8n_Assessment.ipynb`) and **post-read** (`Class6_n8n_postRead.ipynb`): on your Scaler dashboard.
- Finish your mini build, then try a second one. Stretch: add an **error workflow** (Error Trigger → Telegram message to you) and set it in your workflow's Settings → **Error Workflow**.
- **Before the trial ends:** each workflow's **…** menu → **Download**. Classes 7 and 8 use n8n again.
- Next class (7): **What is an AI agent?** Today you chose every step. Next class the AI chooses them.

**Links**
- n8n docs: https://docs.n8n.io
- n8n workflow templates: https://n8n.io/workflows/
- Gemini API key: https://aistudio.google.com/apikey
- Telegram bots: https://core.telegram.org/bots/tutorial
- Hacker News API (used in Challenge A): https://hn.algolia.com/api